# Task A -- one-layer reinitialization and a refitted ensemble weight

Two questions in one session, both about the current best Task A submission (Run 9, the
full-data MuRIL + TF-IDF ensemble at `0.8187`).

1. **Does one-layer reinitialization help Task A?** Task B measured one layer against none
   at +3.0 points, and one against two at +0.5, which is inside noise. Task A has never
   tested either. This notebook runs both `--reinit-layers 1` and `--reinit-layers 2`
   five-fold, so the answer comes from 6,401 out-of-fold rows rather than from Task B.
2. **What is the right blend weight now?** Run 9 uses 57% SVM and 43% MuRIL, fitted in
   Run 8 against a **two-layer** MuRIL. A better MuRIL component should earn more weight,
   so the weight is refitted here rather than assumed.

## Reinitialization does not interfere with the ensemble

The two components are independent. Reinitialization touches only MuRIL's top encoder
layers before fine-tuning; the TF-IDF/SVM is untouched. What it does change is the
*optimal mixing weight*, because that depends on the relative strength of the two
components. Hence question 2.

| setting | value |
|---|---|
| data | all 6,401 deduplicated rows |
| stage 1 | five-fold OOF, split seed 42, for both components |
| stage 2 | full-data fit of the winning arm, five seeds averaged |
| encoder | `google/muril-base-cased`, demojized input |
| epochs / batch | 6 / 8 with 2 accumulation steps, effective 16 |
| blend weight | swept on OOF, then **nested** so the reported gain is honest |
| decision threshold | swept on OOF and nested as well |

## Why the threshold is tuned too

Measured on the TF-IDF floor: threshold 0.5 scores `0.8073`, and a nested threshold
scores `0.8108`, a gain of `+0.0035` for no GPU at all. `hastika.models.muril` already
does this for a single model, but the ensemble notebook hard-codes `> 0.5`, so the blend
has never had it.

## Runtime

About **8.2 hours** with both arms, or 5.5 with one. The budget guard runs the one-layer
arm first, so a short session still answers question 2 and still produces a submission.

Set **Accelerator** to `GPU T4 x2` or `GPU P100` and **Internet** on, then
**Save Version -> Save & Run All**.

In [ ]:
import json, os, pathlib, shutil, subprocess, sys, zipfile

WORK = "/kaggle/working/hastika"
if os.path.isdir(WORK + "/.git"):
    subprocess.run(["git", "-C", WORK, "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", "-q", "-b", "task-b", "--depth", "1",
                    "https://github.com/robinpnalex/Hastika-ICON2026.git", WORK], check=True)
os.chdir(WORK)
os.environ["PYTHONPATH"] = os.path.join(WORK, "src")
sys.path.insert(0, os.path.join(WORK, "src"))
pathlib.Path("artifacts/logs").mkdir(parents=True, exist_ok=True)
print("repo:", os.getcwd())
subprocess.run(["git", "log", "-1", "--oneline"], check=True)
subprocess.run('pip install -q emoji ftfy sentencepiece protobuf "transformers>=4.45,<6"',
               shell=True, check=True)

import numpy as np
import pandas as pd
import torch
from hastika.common.preprocessing import dedupe_index

assert torch.cuda.is_available(), "no GPU -- select a CUDA-enabled runtime"
print("gpu:", torch.cuda.get_device_name(0))
train = pd.read_csv("data/raw/binary_train.csv")
keep = dedupe_index(train["Comment"].tolist(), train["Label"].tolist(), "task A")
print(f"raw labelled rows: {len(train)}; deduplicated rows used for fitting: {len(keep)}")
assert len(keep) == 6401, len(keep)

def run(cmd, log=None):
    print("$", " ".join(cmd), flush=True)
    fh = open(log, "w") if log else None
    p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                         text=True, bufsize=1)
    for line in p.stdout:
        sys.stdout.write(line)
        if fh:
            fh.write(line)
    p.wait()
    if fh:
        fh.close()
    if p.returncode:
        raise RuntimeError(f"exit {p.returncode}: {cmd}")


## 1. Five-fold OOF for both components

The SVM's default mode already produces an OOF matrix. MuRIL runs twice, at one and two
reinitialized layers, on the same split seed so all three matrices line up row for row.

`--select last` is used throughout: `best` picks each fold's checkpoint using the rows it
then reports, which flatters every arm.

In [ ]:
import time
t0 = time.time()
BUDGET_H, RESERVE_MIN = 10.5, 20
left = lambda: BUDGET_H * 3600 - (time.time() - t0) - RESERVE_MIN * 60

SVM_TAG = "task_a_svm_oof"
run([sys.executable, "-u", "-m", "hastika.models.baseline_svm",
     "--task", "a", "--tag", SVM_TAG, "--demojize"],
    log=f"artifacts/logs/{SVM_TAG}.log")

COMMON = ["--model", "google/muril-base-cased", "--folds", "5", "--epochs", "6",
          "--bs", "8", "--grad-accum", "2", "--eval-bs", "32",
          "--select", "last", "--seeds", "42"]
ARMS = [("task_a_reinit1_oof", "1"), ("task_a_reinit2_oof", "2")]
ran = []
for tag, layers in ARMS:
    if left() < 160 * 60:
        print(f"skip {tag}: {left()/60:.0f} min left, needs ~160", flush=True)
        continue
    run([sys.executable, "-u", "-m", "hastika.models.muril", "--tag", tag,
         *COMMON, "--reinit-layers", layers], log=f"artifacts/logs/{tag}.log")
    ran.append((tag, layers))
print("\narms completed:", ran)

## 2. Compare the two reinitialization settings

Both numbers are unbiased out-of-fold macro-F1 over all 6,401 rows, where noise is roughly
0.6 points. Task B could never resolve a gap this small; Task A can.

In [ ]:
from sklearn.metrics import f1_score
from sklearn.model_selection import StratifiedKFold
from hastika.common.preprocessing import clean

df = train.iloc[keep].reset_index(drop=True)
X = np.array([clean(t, demojize=True) for t in df["Comment"]])
y = (df["Label"] == "Hate").astype(int).values

def load_oof(tag):
    return np.load(pathlib.Path("artifacts/runs") / tag / "oof_probs.npy")

svm_oof = load_oof(SVM_TAG)
scores = {"svm": f1_score(y, svm_oof.argmax(1), average="macro")}
for tag, layers in ran:
    scores[f"muril_reinit{layers}"] = f1_score(y, load_oof(tag).argmax(1), average="macro")
for k, v in scores.items():
    print(f"  {k:18s} OOF macro-F1 {v:.4f}")

best_tag, best_layers = max(ran, key=lambda a: f1_score(y, load_oof(a[0]).argmax(1),
                                                        average="macro"))
print(f"\nbetter MuRIL arm: {best_tag} ({best_layers}-layer reinitialization)")

## 3. Refit the blend weight, nested

`w` is the SVM's share. The in-sample sweep is printed for shape, but the number to trust
is the nested one, where `w` and the threshold are chosen on an inner split of each fold's
training rows and applied to rows that never influenced them.

Run 8's value of 0.57 was fitted against a two-layer MuRIL. If the one-layer arm is
stronger, the refitted weight should move toward MuRIL.

In [ ]:
muril_oof = load_oof(best_tag)
GRID_W = np.arange(0.0, 1.01, 0.05)
GRID_T = np.arange(0.30, 0.71, 0.02)

def blend_score(w, t, sv, mu, yy):
    p = w * sv[:, 1] + (1 - w) * mu[:, 1]
    return f1_score(yy, (p > t).astype(int), average="macro")

print("in-sample sweep (optimistic):")
grid = {(w, t): blend_score(w, t, svm_oof, muril_oof, y) for w in GRID_W for t in GRID_T}
(bw, bt), bs = max(grid.items(), key=lambda kv: kv[1])
for w in [0.0, 0.25, 0.43, 0.57, 0.75, 1.0]:
    print(f"  w={w:.2f} at t=0.50  {blend_score(w, 0.50, svm_oof, muril_oof, y):.4f}")
print(f"  best in-sample: w={bw:.2f} t={bt:.2f} -> {bs:.4f}")

pred = np.zeros(len(y), dtype=int)
picks = []
for tr, va in StratifiedKFold(5, shuffle=True, random_state=42).split(X, y):
    itr, iva = next(StratifiedKFold(4, shuffle=True, random_state=7).split(X[tr], y[tr]))
    inner = tr[iva]
    w, t = max(((w, t) for w in GRID_W for t in GRID_T),
               key=lambda p: blend_score(p[0], p[1], svm_oof[inner], muril_oof[inner],
                                         y[inner]))
    picks.append((round(w, 2), round(t, 2)))
    p = w * svm_oof[va, 1] + (1 - w) * muril_oof[va, 1]
    pred[va] = (p > t).astype(int)
nested = f1_score(y, pred, average="macro")
print(f"\nnested blend macro-F1 {nested:.4f}   picks per fold {picks}")
for name, s in scores.items():
    print(f"  vs {name:18s} {nested - s:+.4f}")

W_SVM, THRESH = float(np.mean([w for w, _ in picks])), float(np.mean([t for _, t in picks]))
print(f"\nweights to use for the final fit: SVM {W_SVM:.2f} / MuRIL {1-W_SVM:.2f}, "
      f"threshold {THRESH:.2f}")

## 4. Full-data fit of the winning arm

Five seeds on all 6,401 rows, `--folds 1`, using whichever reinitialization won stage 2.
The SVM is refitted with `--full-fit`. The weight and threshold come from stage 3 and are
not re-optimized here: the hidden validation labels must not touch a final fit.

In [ ]:
SEEDS = ["42", "43", "44", "45", "46"]
FULL_TAG = f"task_a_reinit{best_layers}_full"
SVM_FULL = "task_a_svm_full"

run([sys.executable, "-u", "-m", "hastika.models.baseline_svm",
     "--task", "a", "--tag", SVM_FULL, "--demojize", "--full-fit"],
    log=f"artifacts/logs/{SVM_FULL}.log")
run([sys.executable, "-u", "-m", "hastika.models.muril", "--tag", FULL_TAG,
     "--model", "google/muril-base-cased", "--folds", "1", "--epochs", "6",
     "--bs", "8", "--grad-accum", "2", "--eval-bs", "32", "--select", "last",
     "--reinit-layers", best_layers, "--seeds", *SEEDS],
    log=f"artifacts/logs/{FULL_TAG}.log")

import re
fits = re.findall(r"seed (\d+) FULL FIT, (\d+) rows",
                  pathlib.Path(f"artifacts/logs/{FULL_TAG}.log").read_text())
print(fits)
assert [s for s, _ in fits] == SEEDS, "not all five seeds ran"
assert all(int(n) == 6401 for _, n in fits), "a seed did not see all rows"

## 5. Package the submission

In [ ]:
svm_test = np.load(pathlib.Path("artifacts/runs") / SVM_FULL / "test_probs.npy")
muril_test = np.load(pathlib.Path("artifacts/runs") / FULL_TAG / "test_probs.npy")
ids = pd.read_csv("data/raw/binary_validation_inputs.csv")
p = W_SVM * svm_test[:, 1] + (1 - W_SVM) * muril_test[:, 1]
labels = np.where(p > THRESH, "Hate", "Non-Hate")

out = pathlib.Path("artifacts/runs") / "task_a_reinit_blend"
out.mkdir(parents=True, exist_ok=True)
pd.DataFrame({"id": ids["id"], "label": labels}).to_csv(out / "predictions.csv", index=False)
ZIP = "/kaggle/working/task_a_reinit_blend.zip"
run([sys.executable, "-m", "hastika.common.submission",
     "--task", "a", "--pred", str(out / "predictions.csv"), "--out", ZIP])
with zipfile.ZipFile(ZIP) as f:
    assert f.namelist() == ["predictions.csv"], f.namelist()
print(pd.Series(labels).value_counts().to_dict(),
      f"| weight {W_SVM:.2f}/{1-W_SVM:.2f} | threshold {THRESH:.2f}")

## 6. Preserve outputs

Keep every `oof_probs.npy`. They are what make any future blend or threshold idea testable
in seconds instead of hours, and Task A has never had them stored in the repository.

In [ ]:
OUT = pathlib.Path("/kaggle/working/task_a_reinit_outputs")
OUT.mkdir(parents=True, exist_ok=True)
shutil.copy2(ZIP, OUT / pathlib.Path(ZIP).name)
for tag in [SVM_TAG, SVM_FULL, FULL_TAG] + [t for t, _ in ran]:
    d = pathlib.Path("artifacts/runs") / tag
    for name in ["oof_probs.npy", "test_probs.npy", "predictions.csv"]:
        if (d / name).exists():
            shutil.copy2(d / name, OUT / f"{tag}_{name}")
    log = pathlib.Path(f"artifacts/logs/{tag}.log")
    if log.exists():
        shutil.copy2(log, OUT / log.name)
json.dump({"w_svm": W_SVM, "threshold": THRESH, "nested": nested,
           "oof_scores": scores, "picks": picks}, open(OUT / "blend_summary.json", "w"),
          indent=2)
print(sorted(x.name for x in OUT.iterdir()))

## 7. After CodaBench scores the submission

Record the score in `docs/EXPERIMENTS.md` and `submissions/README.md` against the current
best of `0.8187`, along with the refitted weight and threshold, and the two out-of-fold
reinitialization numbers.

The two out-of-fold numbers from stage 2 are worth recording whatever the leaderboard
says. They are a real local measurement on 6,401 rows, and unlike a CodaBench score on 806
rows they can resolve a gap of well under a point.